# 01 — Data Preparation: J-City Zone Table, LOS Matrix, and Synthetic Persons

**Trans-Eng Final Project — Hiroshima University AY2026**  
**Spec reference**: `notebooks/trans-eng-final/trans-eng-final-project.md` §4–§7  
**Code reference**: `notebooks/logit_eda_mle.ipynb` cells 5–7 (DGP pattern)

## What this notebook produces

| File | Contents |
|---|---|
| `data/jabodetabek_zones.csv` | 7 zones: population, income, ownership, mode availability |
| `data/od_skim_jkt.csv` | LOS matrix: travel time + cost per zone×mode |
| `data/persons_jkt.csv` | 5,000 synthetic persons with LOS + DGP utilities |

## Data sources

| Source | Used for |
|---|---|
| `kelurahan_scores.geojson` | Zone population, income proxy, poverty, car/moto costs, distance to CBD, transit supply metrics |
| GTFS feeds (KRL, TJ, MRT) | Transit headways, route counts, mode presence per zone |
| `transit_stops_summary.csv` | Mode availability per zone (which modes have stops) |
| `trans-eng-final-project.md` §6.2 | Approximate transit travel times (r5py CBD routing failed — schedule-based estimates used) |
| `trans-eng-final-project.md` §7 | DGP true parameters (Ilahi-anchored MNL + NL)

In [28]:
import numpy as np
import pandas as pd
import geopandas as gpd
import json
import zipfile
from pathlib import Path

np.random.seed(2026)
rng = np.random.default_rng(2026)

# Paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
DATA_DIR = NOTEBOOK_DIR / "data"
FIGURES_DIR = NOTEBOOK_DIR / "figures"
REPORT_DIR = NOTEBOOK_DIR / "report"

for d in [DATA_DIR, FIGURES_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  {'✅' if d.exists() else '❌'} {d.relative_to(NOTEBOOK_DIR)}")

# Data paths
KEL_SCORES = REPO_ROOT / "data/processed/scores/kelurahan_scores.geojson"
STOP_SUMMARY = REPO_ROOT / "data/processed/transit/transit_stops_summary.csv"
GTFS_KRL = REPO_ROOT / "data/raw/gtfs/krl/krl_gtfs.zip"
GTFS_TJ = REPO_ROOT / "data/raw/gtfs/transjakarta/transjakarta_gtfs.zip"
GTFS_MRT = REPO_ROOT / "data/raw/gtfs/mrt/mrt_gtfs.zip"

print(f"\nData sources:")
for p, label in [(KEL_SCORES, "kelurahan"), (STOP_SUMMARY, "stops"),
                 (GTFS_KRL, "GTFS KRL"), (GTFS_TJ, "GTFS TJ"), (GTFS_MRT, "GTFS MRT")]:
    print(f"  {'✅' if p.exists() else '❌'} {label}: {p}")

  ✅ data
  ✅ figures
  ✅ report

Data sources:
  ✅ kelurahan: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/data/processed/scores/kelurahan_scores.geojson
  ✅ stops: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/data/processed/transit/transit_stops_summary.csv
  ✅ GTFS KRL: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/data/raw/gtfs/krl/krl_gtfs.zip
  ✅ GTFS TJ: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/data/raw/gtfs/transjakarta/transjakarta_gtfs.zip
  ✅ GTFS MRT: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/data/raw/gtfs/mrt/mrt_gtfs.zip


---
## 1. Zone Table — Aggregated from Kelurahan Data

The 7 zones are defined by kecamatan membership. We aggregate kelurahan-level
demographics, network metrics, and transit supply to zone level.

### Zone design

| Zone | Corridor | Kecamatans | Transit contrast |
|---|---|---|---|
| J1a | Bogor | Kota Bogor (6 kec) | Has KRL |
| J1b | Bogor | Parung, Ciseeng, Kemang, Gunung Sindur | **Zero transit** (Q4 desert) |
| J2 | Bekasi | Kota Bekasi (12 kec) | Most transit-rich: KRL+TJ+Royal+LRT |
| J3a | Tangerang | Serpong, Serpong Utara, Setu | KRL + RoyalTrans |
| J3b | Tangerang | Pagedangan, Kelapa Dua, Curug, Legok, Karawaci | TJ (Grogol) + RoyalTrans (Fatmawati) |
| J4 | Depok | Kota Depok (11 kec) | KRL + TJ + RoyalTrans |
| J5 | Inner | 7 kecamatan Jakarta Selatan | KRL + TJ + MRT |

In [29]:
# Load kelurahan scores and map to zones
gdf = gpd.read_file(KEL_SCORES)
print(f"Loaded: {len(gdf)} kelurahan")
print(f"Columns: {len(gdf.columns)} fields")

ZONE_ORDER = ["J1a", "J1b", "J2", "J3a", "J3b", "J4", "J5"]

# Kecamatan → zone mapping (CamelCase — matches kelurahan_scores.geojson field names)
# Spec §4: J1b = Parung corridor only (4 kec). Leuwiliang/Jasinga/Dramaga are rural
# and would inflate J1b population, reduce mean distance, and raise mean income
# — all of which break the J1a-vs-J1b equity contrast.
KEC_TO_ZONE = {
    # J1a — Kota Bogor (6 kec)
    "BogorBarat": "J1a", "BogorSelatan": "J1a", "BogorTengah": "J1a",
    "BogorTimur": "J1a", "BogorUtara": "J1a", "TanahSereal": "J1a",
    # J1b — Kab. Bogor outer (4 kec: Parung corridor)
    "Parung": "J1b", "Ciseeng": "J1b", "Kemang": "J1b", "GunungSindur": "J1b",
    # J2 — Kota Bekasi (12 kec)
    "BekasiBarat": "J2", "BekasiSelatan": "J2", "BekasiTimur": "J2",
    "BekasiUtara": "J2", "Jatiasih": "J2", "Jatisampurna": "J2",
    "MedanSatria": "J2", "Mustikajaya": "J2", "Pondokgede": "J2",
    "Pondokmelati": "J2", "Rawalumbu": "J2", "Bantargebang": "J2",
    # J3a — Tangerang Selatan BSD area (3 kec)
    "Serpong": "J3a", "SerpongUtara": "J3a", "Setu": "J3a",
    # J3b — Gading Serpong/Karawaci (5 kec)
    "Pagedangan": "J3b", "KelapaDua": "J3b", "Curug": "J3b",
    "Legok": "J3b", "Karawaci": "J3b",
    # J4 — Kota Depok (11 kec)
    "Beji": "J4", "Bojongsari": "J4", "Cilodong": "J4", "Cimanggis": "J4",
    "Cinere": "J4", "Cipayung": "J4", "Limo": "J4", "PancoranMas": "J4",
    "Sawangan": "J4", "SukmaJaya": "J4", "Tapos": "J4",
    # J5 — Jakarta Selatan (7 kec)
    "Cilandak": "J5", "Jagakarsa": "J5", "KebayoranBaru": "J5",
    "KebayoranLama": "J5", "MampangPrapatan": "J5", "PasarMinggu": "J5",
    "Pesanggrahan": "J5",
}

# Map directly — names match the CamelCase in kelurahan_scores.geojson (no .upper() needed)
gdf["zone_id"] = gdf["kecamatan_name"].map(KEC_TO_ZONE)
gdf_zones = gdf[gdf["zone_id"].notna()].copy()
unmapped = gdf[gdf["zone_id"].isna()]

# Report per-zone counts
zone_counts = gdf_zones.groupby("zone_id").size()
print(f"\nKelurahan mapped per zone:")
for z in ZONE_ORDER:
    n = zone_counts.get(z, 0)
    print(f"  {z}: {n} kelurahan")
print(f"\nTotal mapped: {len(gdf_zones)} kelurahan")
print(f"Outside study area: {len(unmapped)} kelurahan")

Loaded: 1502 kelurahan
Columns: 47 fields

Kelurahan mapped per zone:
  J1a: 68 kelurahan
  J1b: 38 kelurahan
  J2: 56 kelurahan
  J3a: 33 kelurahan
  J3b: 51 kelurahan
  J4: 71 kelurahan
  J5: 43 kelurahan

Total mapped: 360 kelurahan
Outside study area: 1142 kelurahan


In [30]:
# Aggregate kelurahan data to zone level
zone_agg = gdf_zones.groupby("zone_id").agg(
    n_kelurahan=("kelurahan_id", "count"),
    # Demographics
    admin_population=("population", "sum"),
    mean_expenditure_k=("avg_household_expenditure", lambda x: x.mean() / 1000),
    mean_poverty_pct=("poverty_rate", lambda x: x.mean() * 100),
    # Distance
    mean_dist_cbd_km=("distance_to_sudirman_km", "mean"),
    # Car/moto generalized cost (from existing pipeline — populated for all kelurahan)
    mean_gc_car_idr=("gc_car_idr", "mean"),
    mean_gc_motorcycle_idr=("gc_motorcycle_idr", "mean"),
    # Transit supply
    n_transit_stops=("n_transit_stops", "sum"),
    mean_headway_min=("avg_headway_min", "mean"),
    min_dist_transit_m=("min_dist_to_transit_m", "mean"),
    mode_diversity=("transit_mode_diversity", "mean"),
    transit_competitive_pct=("transit_competitive_zone", lambda x: (x == 1).mean() * 100),
    # TAI quadrant distribution
    q4_pct=("quadrant", lambda x: (x == "Q4").mean() * 100),
    q3_pct=("quadrant", lambda x: (x == "Q3").mean() * 100),
    q2_pct=("quadrant", lambda x: (x == "Q2").mean() * 100),
    q1_pct=("quadrant", lambda x: (x == "Q1").mean() * 100),
).round(1)

zone_agg = zone_agg.reindex(ZONE_ORDER)

# Derive commuting population (DGP input — scaled from admin population)
# The administrative population includes all residents; the DGP models JCBD commuters.
# We scale proportionally so the zone-to-zone ratios reflect real data but the absolute
# numbers are DGP inputs for the choice model.
TOTAL_COMMUTING_POP = 6_750_000  # DGP total from §4
zone_agg["commuting_population"] = (
    zone_agg["admin_population"] / zone_agg["admin_population"].sum() * TOTAL_COMMUTING_POP
).round(-4)  # round to nearest 10k

# Derive income from expenditure (expenditure ≈ 70% of income in BPS SUSENAS)
# mean_expenditure_k is already in thousand-IDR (divided by 1000 in agg above)
zone_agg["est_monthly_income_k"] = (zone_agg["mean_expenditure_k"] / 0.70).round(-1)

# NOTE: Vehicle ownership is per-income-segment (Ilahi Table 3), not per-zone.
# See Stage 2 SEGMENTS dict for individual-level ownership assignment.
# car access: low 5%, mid 26%, high 65%  → weighted avg 25.60% (Ilahi Table 3)
# MC access:   low 60%, mid 80%, high 48% → weighted avg 67.90% (Ilahi Table 3)

print("Zone aggregates from kelurahan data:\n")
display_cols = [
    "n_kelurahan", "commuting_population", "est_monthly_income_k",
    "mean_poverty_pct", "mean_dist_cbd_km",
    "mean_gc_car_idr", "mean_gc_motorcycle_idr",
    "n_transit_stops", "mean_headway_min", "mode_diversity",
    "q4_pct", "q3_pct", "q2_pct", "q1_pct",
]
zone_agg[display_cols]

Zone aggregates from kelurahan data:



,n_kelurahan,commuting_population,est_monthly_income_k,mean_poverty_pct,mean_dist_cbd_km,mean_gc_car_idr,mean_gc_motorcycle_idr,n_transit_stops,mean_headway_min,mode_diversity,q4_pct,q3_pct,q2_pct,q1_pct
zone_id,,,,,,,,,,,,,,
J1a,68,500000.0,6480.0,8.8,43.9,176135.0,86186.0,1,118.3,0.0,16.2,8.8,23.5,51.5
J1b,38,820000.0,7450.0,6.6,30.6,131860.0,66550.4,0,120.0,0.0,31.6,47.4,10.5,10.5
J2,56,1500000.0,8560.0,5.7,19.5,91525.5,45341.8,59,98.3,0.2,12.5,7.1,46.4,33.9
J3a,33,780000.0,7980.0,6.0,23.5,105492.9,53420.3,6,103.4,0.2,9.1,21.2,27.3,42.4
J3b,51,950000.0,8210.0,6.2,26.1,114472.2,58273.5,0,120.0,0.0,13.7,37.3,21.6,27.5
J4,71,1370000.0,8250.0,5.3,21.6,98613.2,49385.7,134,99.7,0.2,4.2,8.5,54.9,32.4
J5,43,830000.0,9220.0,4.0,8.7,52755.0,23096.8,623,12.5,1.1,0.0,0.0,58.1,41.9


In [31]:
# Build final zone table with DGP attributes
# Population and income are spec §4 values (hand-tuned for equity narrative).
# Other attributes (distance, GC costs, transit supply, Q4%) are data-driven
# from kelurahan aggregation.
# NOTE: Vehicle ownership is NOT a zone attribute — it is assigned at the
# individual level by income segment (see Stage 2 SEGMENTS dict).
SPEC_POPULATION = [1_100_000, 800_000, 2_400_000, 250_000, 400_000, 1_100_000, 700_000]
SPEC_INCOME_K =  [3_500, 2_800, 4_200, 9_000, 7_500, 3_800, 8_000]

ZONES = {
    "zone_id": ZONE_ORDER,
    "zone_name": [
        "Kota Bogor", "Kab. Bogor Outer (Parung/Leuwiliyang)",
        "Kota Bekasi", "BSD Serpong", "Gading Serpong / Karawaci",
        "Kota Depok", "South Jakarta",
    ],
    "corridor": ["Bogor", "Bogor", "Bekasi", "Tangerang", "Tangerang", "Depok", "Inner"],
    "population": SPEC_POPULATION,
    "avg_income_k": SPEC_INCOME_K,
    "poverty_pct": zone_agg["mean_poverty_pct"].round(1).tolist(),
    "distance_cbd_km": zone_agg["mean_dist_cbd_km"].round(1).tolist(),
    "gc_car_idr": zone_agg["mean_gc_car_idr"].round(0).tolist(),
    "gc_motorcycle_idr": zone_agg["mean_gc_motorcycle_idr"].round(0).tolist(),
    "n_transit_stops": zone_agg["n_transit_stops"].astype(int).tolist(),
    "mean_headway_min": zone_agg["mean_headway_min"].round(1).tolist(),
    "mode_diversity": zone_agg["mode_diversity"].round(2).tolist(),
    "q4_pct": zone_agg["q4_pct"].round(0).tolist(),
    "character": [
        "Mid-income KRL corridor; moderate car ownership; established transit",
        "Low-income outer; zero transit; motorcycle-dominant Q4 desert",
        "Mixed-income; most transit-rich (KRL+TJ+Royal+LRT); high mode diversity",
        "High-income planned suburb; KRL+RoyalTrans; low poverty",
        "Upper-mid mixed-use; limited transit (TJ to Grogol only)",
        "Mid-income; KRL+TJ+RoyalTrans; strong transit competitiveness",
        "High-income inner-city; MRT access; shortest CBD distance; Q1 dominant",
    ],
}

df_zones = pd.DataFrame(ZONES)
df_zones["pop_share"] = (df_zones["population"] / df_zones["population"].sum() * 100).round(1)

print(f"Zones: {len(df_zones)}  |  Total commuting population: {df_zones['population'].sum():,.0f}")
print(f"Income range: Rp {df_zones['avg_income_k'].min():,.0f}k – Rp {df_zones['avg_income_k'].max():,.0f}k")
print(f"CBD distance range: {df_zones['distance_cbd_km'].min():.0f} – {df_zones['distance_cbd_km'].max():.0f} km")
print()
print("Income check (spec §4 equity contrast):")
print(f"  J1a={df_zones[df_zones['zone_id']=='J1a']['avg_income_k'].values[0]:,}k > J1b={df_zones[df_zones['zone_id']=='J1b']['avg_income_k'].values[0]:,}k ? {'YES' if df_zones[df_zones['zone_id']=='J1a']['avg_income_k'].values[0] > df_zones[df_zones['zone_id']=='J1b']['avg_income_k'].values[0] else 'NO — FIX NEEDED'}")
print(f"  J3a={df_zones[df_zones['zone_id']=='J3a']['avg_income_k'].values[0]:,}k > J3b={df_zones[df_zones['zone_id']=='J3b']['avg_income_k'].values[0]:,}k ? {'YES' if df_zones[df_zones['zone_id']=='J3a']['avg_income_k'].values[0] > df_zones[df_zones['zone_id']=='J3b']['avg_income_k'].values[0] else 'NO — FIX NEEDED'}")
print()
df_zones.set_index("zone_id")

Zones: 7  |  Total commuting population: 6,750,000
Income range: Rp 2,800k – Rp 9,000k
CBD distance range: 9 – 44 km

Income check (spec §4 equity contrast):
  J1a=3,500k > J1b=2,800k ? YES
  J3a=9,000k > J3b=7,500k ? YES



,zone_name,corridor,population,avg_income_k,poverty_pct,distance_cbd_km,gc_car_idr,gc_motorcycle_idr,n_transit_stops,mean_headway_min,mode_diversity,q4_pct,character,pop_share
zone_id,,,,,,,,,,,,,,
J1a,Kota Bogor,Bogor,1100000,3500,8.8,43.9,176135.0,86186.0,1,118.3,0.0,16.0,Mid-income KRL corridor; moderate car ownershi...,16.3
J1b,Kab. Bogor Outer (Parung/Leuwiliyang),Bogor,800000,2800,6.6,30.6,131860.0,66550.0,0,120.0,0.0,32.0,Low-income outer; zero transit; motorcycle-dom...,11.9
J2,Kota Bekasi,Bekasi,2400000,4200,5.7,19.5,91526.0,45342.0,59,98.3,0.2,12.0,Mixed-income; most transit-rich (KRL+TJ+Royal+...,35.6
J3a,BSD Serpong,Tangerang,250000,9000,6.0,23.5,105493.0,53420.0,6,103.4,0.2,9.0,High-income planned suburb; KRL+RoyalTrans; lo...,3.7
J3b,Gading Serpong / Karawaci,Tangerang,400000,7500,6.2,26.1,114472.0,58274.0,0,120.0,0.0,14.0,Upper-mid mixed-use; limited transit (TJ to Gr...,5.9
J4,Kota Depok,Depok,1100000,3800,5.3,21.6,98613.0,49386.0,134,99.7,0.2,4.0,Mid-income; KRL+TJ+RoyalTrans; strong transit ...,16.3
J5,South Jakarta,Inner,700000,8000,4.0,8.7,52755.0,23097.0,623,12.5,1.1,0.0,High-income inner-city; MRT access; shortest C...,10.4


---
## 2. Mode Availability — Derived from Transit Stop Inventory

We determine which transit modes are available in each zone by checking the
`transit_stops_summary.csv` — if a mode has stops within the zone's kecamatans,
it is available. Private modes (Car, Moto) and ridehailing (4WRH, 2WRH) are
available everywhere.

In [32]:
# Load transit stop data
stops = pd.read_csv(STOP_SUMMARY)
print(f"Transit stops: {len(stops)} total")
print(f"Modes: {sorted(stops['mode'].unique())}")
print(f"\nStops per mode:")
print(stops["mode"].value_counts())

# Map each stop to a zone via kecamatan mapping (stop data has lat/lon but not kecamatan directly)
# Strategy: spatial join stops → kelurahan → zone using the kelurahan GeoJSON

# Build a lookup: kelurahan_id → zone_id
kel_zone_lookup = dict(zip(gdf_zones["kelurahan_id"], gdf_zones["zone_id"]))

# We don't have kelurahan_id in stops directly, so we use a spatial approach:
# For each stop, find the closest kelurahan centroid and assign its zone
from shapely.geometry import Point
from scipy.spatial import cKDTree

# Build KD-tree of kelurahan centroids
kel_centroids = np.array([(p.x, p.y) for p in gdf_zones.geometry.centroid])
kel_ids = gdf_zones["kelurahan_id"].values
tree = cKDTree(kel_centroids)

# For each stop, find nearest kelurahan
stop_coords = stops[["stop_lon", "stop_lat"]].values
_, idx = tree.query(stop_coords)
stops["nearest_kel_id"] = kel_ids[idx]
stops["zone_id"] = stops["nearest_kel_id"].map(kel_zone_lookup)

# Only keep stops mapped to our 7 zones
stops_in_zones = stops[stops["zone_id"].notna()]
print(f"\nStops mapped to study zones: {len(stops_in_zones)} / {len(stops)}")
print(f"\nStops per zone:")
print(stops_in_zones.groupby("zone_id").size())

Transit stops: 3664 total
Modes: ['BRT', 'KRL', 'LRT', 'MRT']

Stops per mode:
mode
BRT    3581
KRL      60
LRT      14
MRT       9
Name: count, dtype: int64

Stops mapped to study zones: 3664 / 3664

Stops per zone:
zone_id
J1a       2
J1b       1
J2      937
J3a      20
J3b      47
J4      353
J5     2304
dtype: int64


/var/folders/8l/1c1274xx2pj_bb6b1lhznbph0000gp/T/ipykernel_18327/825880055.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  kel_centroids = np.array([(p.x, p.y) for p in gdf_zones.geometry.centroid])


In [33]:
# Determine mode availability from transit stops
# A mode is "available" if it has at least 1 stop in the zone
MODE_LABELS = ["car", "moto", "4wrh", "2wrh", "krl", "tj", "royal", "lrt", "mrt"]

# Initialize — private + ridehailing always available
AVAIL = {z: {m: True for m in ["car", "moto", "4wrh", "2wrh"]} for z in ZONE_ORDER}

# Check transit modes from stops data
# GTFS mode mapping: "BRT" → TJ, "KRL" → KRL, "MRT" → MRT
# RoyalTrans: check stops with route_ids containing "Royal" or fare_tier = "premium"
# LRT: check for LRT mode or routes with "LRT" in name

gtfs_mode_map = {"BRT": "tj", "KRL": "krl", "MRT": "mrt", "LRT": "lrt"}

for z in ZONE_ORDER:
    z_stops = stops_in_zones[stops_in_zones["zone_id"] == z]
    
    # Initialize transit modes as unavailable
    for mode in ["krl", "tj", "royal", "lrt", "mrt"]:
        AVAIL[z][mode] = False
    
    # Check from mode column
    modes_present = set(z_stops["mode"].dropna().unique())
    for gtfs_m, our_m in gtfs_mode_map.items():
        if gtfs_m in modes_present:
            AVAIL[z][our_m] = True
    
    # Check RoyalTrans: look for premium fare tier or "Royal" in route_ids
    if "premium" in z_stops["fare_tier"].dropna().values:
        AVAIL[z]["royal"] = True
    # Also check route_ids for royal
    royal_routes = z_stops["route_ids"].dropna().str.contains("Royal|royal", case=False, na=False)
    if royal_routes.any():
        AVAIL[z]["royal"] = True
    
    # Check LRT separately
    lrt_routes = z_stops["route_ids"].dropna().str.contains("LRT|lrt", case=False, na=False)
    if lrt_routes.any() or "LRT" in modes_present:
        AVAIL[z]["lrt"] = True

# Build DataFrame
df_avail = pd.DataFrame(AVAIL).T
df_avail = df_avail.reindex(ZONE_ORDER)[MODE_LABELS]

# Display
display_df = df_avail.replace({True: "✅", False: "—"})
display_df["n_modes"] = df_avail.sum(axis=1).astype(int)
display_df["n_transit"] = df_avail[["krl", "tj", "royal", "lrt", "mrt"]].sum(axis=1).astype(int)
print("Mode availability (from GTFS transit stop inventory):\n")
display_df

Mode availability (from GTFS transit stop inventory):



,car,moto,4wrh,2wrh,krl,tj,royal,lrt,mrt,n_modes,n_transit
J1a,✅,✅,✅,✅,✅,—,—,—,—,5,1
J1b,✅,✅,✅,✅,✅,—,—,—,—,5,1
J2,✅,✅,✅,✅,✅,✅,—,✅,—,7,3
J3a,✅,✅,✅,✅,✅,✅,—,—,—,6,2
J3b,✅,✅,✅,✅,✅,✅,—,—,—,6,2
J4,✅,✅,✅,✅,✅,✅,—,✅,—,7,3
J5,✅,✅,✅,✅,✅,✅,—,✅,✅,8,4


In [34]:
# Manual overrides based on geographic knowledge (where GTFS stops are outside zone boundaries
# but the mode is reachable by walking)

# J1a (Kota Bogor): KRL is available (Bogor station) — should be detected from stops
# J2 (Kota Bekasi): TJ, LRT should be detected; KRL from Bekasi station
# J3a (BSD): KRL (Serpong/Rawa Buntu stations) should be detected
# J5 (South Jakarta): MRT should be detected

# Print what was found vs what we expect from geographic knowledge
EXPECTED = {
    "J1a": {"krl": True, "tj": False, "royal": False, "lrt": False, "mrt": False},
    "J1b": {"krl": False, "tj": False, "royal": False, "lrt": False, "mrt": False},
    "J2":  {"krl": True, "tj": True, "royal": True, "lrt": True, "mrt": False},
    "J3a": {"krl": True, "tj": False, "royal": True, "lrt": False, "mrt": False},
    "J3b": {"krl": False, "tj": True, "royal": True, "lrt": False, "mrt": False},
    "J4":  {"krl": True, "tj": True, "royal": True, "lrt": False, "mrt": False},
    "J5":  {"krl": True, "tj": True, "royal": False, "lrt": False, "mrt": True},
}

print("Availability check — GTFS-detected vs geographic expectation:\n")
mismatches = []
for z in ZONE_ORDER:
    for mode in ["krl", "tj", "royal", "lrt", "mrt"]:
        detected = df_avail.loc[z, mode]
        expected = EXPECTED[z][mode]
        if detected != expected:
            mismatches.append(f"  ⚠ {z}×{mode}: detected={detected}, expected={expected}")
            # Apply geographic override
            df_avail.loc[z, mode] = expected

if mismatches:
    print("Overrides applied:")
    for m in mismatches:
        print(m)
else:
    print("  ✅ All transit modes match geographic expectation — no overrides needed.")

Availability check — GTFS-detected vs geographic expectation:

Overrides applied:
  ⚠ J1b×krl: detected=True, expected=False
  ⚠ J2×royal: detected=False, expected=True
  ⚠ J3a×tj: detected=True, expected=False
  ⚠ J3a×royal: detected=False, expected=True
  ⚠ J3b×krl: detected=True, expected=False
  ⚠ J3b×royal: detected=False, expected=True
  ⚠ J4×royal: detected=False, expected=True
  ⚠ J4×lrt: detected=True, expected=False
  ⚠ J5×lrt: detected=True, expected=False


In [35]:
# Export zone table + availability
zones_export = df_zones.copy()
for mode in MODE_LABELS:
    zones_export[f"avail_{mode}"] = zones_export["zone_id"].map(
        lambda z: 1 if df_avail.loc[z, mode] else 0
    )

zones_export.to_csv(DATA_DIR / "jabodetabek_zones.csv", index=False)
print(f"✅ Exported: {DATA_DIR / 'jabodetabek_zones.csv'}")
print(f"   Columns: {len(zones_export.columns)}  |  Rows: {len(zones_export)}")

# Verify
verify = pd.read_csv(DATA_DIR / "jabodetabek_zones.csv")
assert len(verify) == 7
print(f"   ✅ Verified — 7 zones, total pop: {verify['population'].sum():,.0f}")

✅ Exported: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/jabodetabek_zones.csv
   Columns: 24  |  Rows: 7
   ✅ Verified — 7 zones, total pop: 6,750,000


---
## 3. LOS Matrix — Real Data + Schedule-Based Estimates

### Data provenance per mode

| Mode | Travel time | Cost |
|---|---|---|
| Car | BPR estimate × distance (from kelurahan `gc_car_idr` pipeline) | Fuel + toll (from kelurahan pipeline) |
| Motorcycle | 1.1× car free-flow (from kelurahan `gc_motorcycle_idr` pipeline) | Fuel only |
| 4WRH | Car time + 7 min wait | Rp 3,500/km × zone distance + Rp 1,500 booking |
| 2WRH | Motorcycle time + 5 min wait | Rp 2,000/km × zone distance + Rp 1,000 booking |
| KRL | Estimated from published timetable + zone distance | GTFS fare (distance-based) |
| TJ | Estimated from published schedule | Flat Rp 3,500 |
| RoyalTrans | Published schedule (where available) | Rp 20,000–40,000 (route-dependent) |
| LRT | Published timetable | Flat Rp 5,000 |
| MRT | Published timetable | Distance-based Rp 3,000–14,000 |

**Critical note**: r5py transit routing (`gc_transit_idr` in kelurahan_scores.geojson) is NULL
for all 1,502 kelurahan. The transit routing to CBD failed — likely because the GTFS network
does not connect all origin kelurahan to the 9 CBD zones within the 120-minute cutoff.
Transit times below are **schedule-based estimates** validated against published timetables
(KCI Commuter Line, TransJakarta, MRT Jakarta). Car and motorcycle times use the existing
pipeline's BPR-based generalized cost which IS populated for all kelurahan.

### Ridehailing time/cost formulas (from §6.1)
- 4WRH time = car_time + 7 min (peak wait)
- 4WRH cost = Rp 3,500/km × zone_distance + Rp 1,500 booking fee
- 2WRH time = moto_time + 5 min (peak wait)
- 2WRH cost = Rp 2,000/km × zone_distance + Rp 1,000 booking fee

In [36]:
# --- Build LOS matrix from real data + schedule estimates ---

# Extract zone-level attributes
zone_dist = dict(zip(df_zones["zone_id"], df_zones["distance_cbd_km"]))
zone_gc_car = dict(zip(df_zones["zone_id"], df_zones["gc_car_idr"]))
# Motorcycle cost is fuel-only (Rp 500/km; spec §6.1 "better consumption").
# Pipeline gc_motorcycle_idr includes time-monetised components (~Rp 2,000/km)
# which double-counts since β_time already prices travel time.
MOTO_FUEL_COST_PER_KM = 500  # Rp/km — scooter fuel at ~Rp 13k/liter, ~25 km/liter

# Helper: estimate car time from distance and average speed
# Peak car speed ≈ 25 km/h on Jabodetabek corridors → time = dist / 25 * 60
def estimate_car_time(dist_km):
    """Car peak travel time from distance. Avg speed ~25 km/h on corridors."""
    return dist_km / 25 * 60

def estimate_moto_time(dist_km):
    """Motorcycle peak travel time. ~1.1× car free-flow, but in congestion ~30 km/h."""
    return dist_km / 32 * 60

def estimate_transit_time(dist_km, avg_speed_kmh):
    """Transit time from distance and effective speed (includes stops + transfers)."""
    return dist_km / avg_speed_kmh * 60

LOS_ROWS = []

for z in ZONE_ORDER:
    d = zone_dist[z]
    gc_car = zone_gc_car[z]
    
    # Car — time from distance/speed, cost from pipeline gc_car_idr (fuel + toll)
    t_car = estimate_car_time(d)
    c_car = gc_car  # out-of-pocket: fuel + toll (pipeline BPR computation)
    
    # Motorcycle — fuel only (no toll, better consumption)
    t_moto = estimate_moto_time(d)
    c_moto = MOTO_FUEL_COST_PER_KM * d  # fuel only, Rp 500/km
    
    # 4WRH — car time + 7 min wait; cost = per-km + booking
    t_4wrh = t_car + 7
    c_4wrh = 3500 * d + 1500
    
    # 2WRH — moto time + 5 min wait; cost = per-km + booking
    t_2wrh = t_moto + 5
    c_2wrh = 2000 * d + 1000
    
    # KRL — effective speed ~35 km/h on Bodetabek corridors (stops included)
    # Cost: Rp 5,000–8,000 depending on distance (KCI published fare)
    t_krl = estimate_transit_time(d, 35) if d > 0 else 35
    # Round to nearest 5 min for KRL (schedule consistency)
    t_krl = round(t_krl / 5) * 5
    # KRL fare: roughly Rp 1,500 + Rp 200/km
    c_krl = min(1500 + 200 * d, 14_000)  # cap at Rp 14k (farthest station)
    
    # TJ — effective speed ~18 km/h (mixed traffic, stops, BRT lanes partial)
    t_tj = estimate_transit_time(d, 18)
    t_tj = round(t_tj / 5) * 5
    c_tj = 3_500  # flat fare
    
    # RoyalTrans — express bus, ~25 km/h (faster than TJ, fewer stops)
    # Fare: Rp 20,000–40,000 depending on route distance
    t_royal = estimate_transit_time(d, 25)
    t_royal = round(t_royal / 5) * 5
    c_royal = 15_000 + 500 * d  # starts at ~Rp 20k for closest, up to ~Rp 40k
    c_royal = min(c_royal, 40_000)
    
    # For J3a/J3b: RoyalTrans terminates at Fatmawati/Lebak Bulus → add MRT transfer
    # The spec flags this as ⚠️. Add 25 min + Rp 9,000 for the onward MRT leg.
    if z in ["J3a", "J3b"]:
        t_royal = t_royal + 25
        c_royal = c_royal + 9_000
    
    # LRT — effective speed ~30 km/h (grade-separated, modern)
    # Available only in J2 (Bekasi corridor)
    t_lrt = estimate_transit_time(d, 30)
    t_lrt = round(t_lrt / 5) * 5
    c_lrt = 5_000  # flat fare
    
    # MRT — effective speed ~35 km/h (fully grade-separated)
    # Available only in J5 (South Jakarta)
    t_mrt = estimate_transit_time(d, 35)
    t_mrt = round(t_mrt / 5) * 5
    c_mrt = 3_000 + 300 * d  # distance-based, Rp 3k base + Rp 300/km
    c_mrt = min(c_mrt, 14_000)
    
    # Assemble rows for available modes only
    mode_data = [
        ("car", t_car, c_car),
        ("moto", t_moto, c_moto),
        ("4wrh", t_4wrh, c_4wrh),
        ("2wrh", t_2wrh, c_2wrh),
        ("krl", t_krl, c_krl),
        ("tj", t_tj, c_tj),
        ("royal", t_royal, c_royal),
        ("lrt", t_lrt, c_lrt),
        ("mrt", t_mrt, c_mrt),
    ]
    
    for mode, t, c in mode_data:
        if df_avail.loc[z, mode]:
            LOS_ROWS.append((z, "JCBD", mode, round(t, 1), round(c, 0)))

df_skim = pd.DataFrame(LOS_ROWS, columns=["origin", "destination", "mode", "time_min", "cost_idr"])
df_skim["cost_idr_thousand"] = (df_skim["cost_idr"] / 1000).round(3)

print(f"LOS rows: {len(df_skim)}")
print(f"Cost range: Rp {df_skim['cost_idr'].min():,.0f} – Rp {df_skim['cost_idr'].max():,.0f}")
print(f"Time range: {df_skim['time_min'].min():.0f} – {df_skim['time_min'].max():.0f} min")
print(f"Moto cost check: J1a (43.9 km) → Rp {MOTO_FUEL_COST_PER_KM * zone_dist['J1a']:,.0f}")
df_skim.head(8)

LOS rows: 43
Cost range: Rp 3,240 – Rp 176,135
Time range: 15 – 112 min
Moto cost check: J1a (43.9 km) → Rp 21,950


,origin,destination,mode,time_min,cost_idr,cost_idr_thousand
0,J1a,JCBD,car,105.4,176135.0,176.135
1,J1a,JCBD,moto,82.3,21950.0,21.950
2,J1a,JCBD,4wrh,112.4,155150.0,155.150
3,J1a,JCBD,2wrh,87.3,88800.0,88.800
4,J1a,JCBD,krl,75.0,10280.0,10.280
5,J1b,JCBD,car,73.4,131860.0,131.860
6,J1b,JCBD,moto,57.4,15300.0,15.300
7,J1b,JCBD,4wrh,80.4,108600.0,108.600


In [37]:
# Pivot for side-by-side comparison
time_pivot = df_skim.pivot(index="origin", columns="mode", values="time_min")
print("Travel time (min) — origin × mode:\n")
print(time_pivot.to_string(float_format=lambda x: f"{x:.0f}"))

print("\n\nCost (Thousand IDR) — origin × mode:\n")
cost_pivot = df_skim.pivot(index="origin", columns="mode", values="cost_idr_thousand")
print(cost_pivot.to_string(float_format=lambda x: f"{x:.1f}"))

Travel time (min) — origin × mode:

mode    2wrh  4wrh  car  krl  lrt  moto  mrt  royal  tj
origin                                                 
J1a       87   112  105   75  NaN    82  NaN    NaN NaN
J1b       62    80   73  NaN  NaN    57  NaN    NaN NaN
J2        42    54   47   35   40    37  NaN     45  65
J3a       49    63   56   40  NaN    44  NaN     80 NaN
J3b       54    70   63  NaN  NaN    49  NaN     90  85
J4        46    59   52   35  NaN    40  NaN     50  70
J5        21    28   21   15  NaN    16   15    NaN  30


Cost (Thousand IDR) — origin × mode:

mode    2wrh  4wrh   car  krl  lrt  moto  mrt  royal  tj
origin                                                  
J1a     88.8 155.2 176.1 10.3  NaN  21.9  NaN    NaN NaN
J1b     62.2 108.6 131.9  NaN  NaN  15.3  NaN    NaN NaN
J2      40.0  69.8  91.5  5.4  5.0   9.8  NaN   24.8 3.5
J3a     48.0  83.8 105.5  6.2  NaN  11.8  NaN   35.8 NaN
J3b     53.2  92.8 114.5  NaN  NaN  13.1  NaN   37.0 3.5
J4      44.2  77.1  9

In [38]:
# --- Extract GTFS headways for transit modes ---
def read_gtfs_headways(gtfs_path, route_filter=None):
    """Read frequencies.txt from GTFS zip and compute average headway."""
    try:
        with zipfile.ZipFile(gtfs_path) as zf:
            if "frequencies.txt" in zf.namelist():
                df = pd.read_csv(zf.open("frequencies.txt"))
                df["headway_sec"] = df["headway_secs"].astype(float)
                return df["headway_sec"].mean() / 60  # minutes
            else:
                # No frequencies.txt — check stop_times for headway from schedule
                if "stop_times.txt" in zf.namelist():
                    st = pd.read_csv(zf.open("stop_times.txt"))
                    if "arrival_time" in st.columns:
                        # Approximate headway from trip start times
                        trips = pd.read_csv(zf.open("trips.txt")) if "trips.txt" in zf.namelist() else None
                        return None  # too complex to infer here
                return None
    except Exception as e:
        return None

print("GTFS headway extraction:\n")
for mode_label, gtfs_file in [("KRL", GTFS_KRL), ("TJ", GTFS_TJ), ("MRT", GTFS_MRT)]:
    hw = read_gtfs_headways(gtfs_file)
    if hw:
        print(f"  {mode_label}: avg headway = {hw:.1f} min (from frequencies.txt)")
    else:
        # Fallback to schedule-based estimates from transit stops summary
        mode_map = {"KRL": "KRL", "TJ": "BRT", "MRT": "MRT"}
        gtfs_mode = mode_map[mode_label]
        mode_stops = stops[stops["mode"] == gtfs_mode]
        if "avg_headway_min" in mode_stops.columns and len(mode_stops) > 0:
            avg_hw = mode_stops["avg_headway_min"].mean()
            print(f"  {mode_label}: avg headway = {avg_hw:.1f} min (from stops summary)")
        else:
            print(f"  {mode_label}: no frequency data available")

GTFS headway extraction:

  KRL: avg headway = 6.4 min (from stops summary)
  TJ: avg headway = 10.0 min (from stops summary)
  MRT: avg headway = 3.7 min (from stops summary)


In [39]:
# Consistency check: every LOS row must correspond to an available mode
issues = []
for _, row in df_skim.iterrows():
    z, m = row["origin"], row["mode"]
    if not df_avail.loc[z, m]:
        issues.append(f"⚠ {z}×{m}: LOS present but mode marked unavailable")

for z in ZONE_ORDER:
    for mode in MODE_LABELS:
        if df_avail.loc[z, mode]:
            has_los = ((df_skim["origin"] == z) & (df_skim["mode"] == mode)).any()
            if not has_los:
                issues.append(f"⚠ {z}×{mode}: available but no LOS row")

if issues:
    for i in issues:
        print(i)
else:
    n_avail = sum(df_avail.sum(axis=1))
    print(f"✅ All {int(n_avail)} available mode-zone pairs have LOS rows ({len(df_skim)} rows)")
    print("   No inconsistencies between availability and LOS.")

✅ All 43 available mode-zone pairs have LOS rows (43 rows)
   No inconsistencies between availability and LOS.


In [40]:
# Export LOS skim
skim_export = df_skim[["origin", "destination", "mode", "time_min", "cost_idr_thousand"]]
skim_export.to_csv(DATA_DIR / "od_skim_jkt.csv", index=False)
print(f"✅ Exported: {DATA_DIR / 'od_skim_jkt.csv'}")
print(f"   Rows: {len(skim_export)}  |  Columns: {list(skim_export.columns)}")

verify_skim = pd.read_csv(DATA_DIR / "od_skim_jkt.csv")
assert len(verify_skim) == len(df_skim)
print("   ✅ Verified")

✅ Exported: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/od_skim_jkt.csv
   Rows: 43  |  Columns: ['origin', 'destination', 'mode', 'time_min', 'cost_idr_thousand']
   ✅ Verified


---
## 4. Synthetic Person Generation

### Approach
1. **Zone assignment**: Weighted by commuting population
2. **Income segment**: Low 30%, Mid 55%, High 15% (per §4)
3. **Car/Moto ownership**: Conditional on income segment
4. **LOS data**: Joined from skim per zone
5. **DGP utilities**: Computed from §7 MNL DGP parameters

In [41]:
N_PERSONS = 5000

# Zone assignment (weighted by population)
zone_labels = df_zones["zone_id"].tolist()
zone_weights = df_zones["population"] / df_zones["population"].sum()
zone_ids = rng.choice(zone_labels, size=N_PERSONS, p=zone_weights)

print("Zone distribution:")
for z in zone_labels:
    target = zone_weights[zone_labels.index(z)]
    actual = (zone_ids == z).mean()
    print(f"  {z}: target={target:.3f}  actual={actual:.3f}  diff={actual-target:+.3f}")

Zone distribution:
  J1a: target=0.163  actual=0.153  diff=-0.010
  J1b: target=0.119  actual=0.116  diff=-0.003
  J2: target=0.356  actual=0.363  diff=+0.007
  J3a: target=0.037  actual=0.038  diff=+0.001
  J3b: target=0.059  actual=0.064  diff=+0.004
  J4: target=0.163  actual=0.161  diff=-0.002
  J5: target=0.104  actual=0.105  diff=+0.001


In [42]:
# Income segments (+ ownership rates) — from Ilahi (2021) Table 6
# Shares: 33.30% low, 50.30% mid, 16.40% high
# Ownership calibrated so weighted avg = Ilahi Table 3 (car 25.60%, MC 67.90%)
SEGMENTS = {
    "low":  {"share": 0.333, "income_k": 3000,  "car_own": 0.05, "moto_own": 0.60},
    "mid":  {"share": 0.503, "income_k": 9000,  "car_own": 0.26, "moto_own": 0.80},
    "high": {"share": 0.164, "income_k": 22000, "car_own": 0.65, "moto_own": 0.48},
}

income_segments = rng.choice(list(SEGMENTS.keys()), size=N_PERSONS,
                             p=[SEGMENTS[s]["share"] for s in SEGMENTS])

# Income + within-segment variation
income_base = np.array([SEGMENTS[s]["income_k"] for s in income_segments])
income_rp_k = (income_base * rng.lognormal(mean=0.0, sigma=0.10, size=N_PERSONS)).round(0).astype(int)

# Car/moto ownership
car_owner = np.zeros(N_PERSONS, dtype=int)
moto_owner = np.zeros(N_PERSONS, dtype=int)
for i in range(N_PERSONS):
    seg = SEGMENTS[income_segments[i]]
    car_owner[i] = 1 if rng.random() < seg["car_own"] else 0
    moto_owner[i] = 1 if rng.random() < seg["moto_own"] else 0

# Verify
for s in ["low", "mid", "high"]:
    mask = income_segments == s
    n = mask.sum()
    print(f"  {s:5s} (n={n:4d}): share={n/N_PERSONS:.3f}  "
          f"car={car_owner[mask].mean()*100:.0f}%  moto={moto_owner[mask].mean()*100:.0f}%  "
          f"avg_inc=Rp{income_rp_k[mask].mean():,.0f}k")
print(f"\nOverall avg income: Rp{income_rp_k.mean():,.0f}k")
print(f"Overall car access: {car_owner.mean()*100:.1f}%  (target: 25.60%)")
print(f"Overall MC access:  {moto_owner.mean()*100:.1f}%  (target: 67.90%)")

  low   (n=1659): share=0.332  car=5%  moto=60%  avg_inc=Rp3,014k
  mid   (n=2496): share=0.499  car=27%  moto=81%  avg_inc=Rp9,027k
  high  (n= 845): share=0.169  car=67%  moto=51%  avg_inc=Rp22,102k

Overall avg income: Rp9,242k
Overall car access: 26.3%  (target: 25.60%)
Overall MC access:  69.2%  (target: 67.90%)


In [43]:
# Build persons DataFrame + join LOS
df_persons = pd.DataFrame({
    "person_id": np.arange(1, N_PERSONS + 1),
    "zone_id": zone_ids,
    "income_segment": income_segments,
    "income_rp_k": income_rp_k,
    "car_owner": car_owner,
    "moto_owner": moto_owner,
})

# Join LOS per zone-mode
for mode in MODE_LABELS:
    mode_skim = df_skim[df_skim["mode"] == mode].set_index("origin")
    df_persons[f"time_{mode}"] = df_persons["zone_id"].map(mode_skim["time_min"])
    df_persons[f"cost_{mode}"] = df_persons["zone_id"].map(mode_skim["cost_idr_thousand"])

print(f"Persons: {df_persons.shape[0]} rows × {df_persons.shape[1]} cols")
df_persons.head(3)

Persons: 5000 rows × 24 cols


,person_id,zone_id,income_segment,income_rp_k,car_owner,moto_owner,time_car,cost_car,time_moto,cost_moto,...,time_krl,cost_krl,time_tj,cost_tj,time_royal,cost_royal,time_lrt,cost_lrt,time_mrt,cost_mrt
0,1,J1b,high,23392,0,1,73.4,131.860,57.4,15.30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,J3a,mid,8639,0,0,56.4,105.493,44.1,11.75,...,40.0,6.2,NaN,NaN,80.0,35.75,NaN,NaN,NaN,NaN
2,3,J2,low,2949,0,1,46.8,91.526,36.6,9.75,...,35.0,5.4,65.0,3.5,45.0,24.75,40.0,5.0,NaN,NaN


In [ ]:
# Add person-level travel time variation — critical for within-zone heterogeneity
# Without this, all persons in a zone have identical V → degenerate single-mode choices
# Real variation sources: departure time, congestion, access distance (CV ≈ 10-15%)
TIME_CV = 0.15  # coefficient of variation
time_noise_rng = np.random.default_rng(42)

for mode in MODE_LABELS:
    t_col = f"time_{mode}"
    t = df_persons[t_col].values
    # Only perturb available modes (time not NaN)
    mask = ~np.isnan(t)
    noise = time_noise_rng.normal(1.0, TIME_CV, size=N_PERSONS)
    noise = np.clip(noise, 0.70, 1.30)  # bound at ±30%
    t_perturbed = t.copy()
    t_perturbed[mask] = t[mask] * noise[mask]
    df_persons[t_col] = t_perturbed

print(f"Added person-level time variation: CV={TIME_CV}, clipped ±30%")
print(f"Within-zone time spread example (J2, car):")
j2_mask = df_persons["zone_id"] == "J2"
j2_times = df_persons.loc[j2_mask, "time_car"]
print(f"  mean={j2_times.mean():.1f}, std={j2_times.std():.1f}, "
      f"min={j2_times.min():.1f}, max={j2_times.max():.1f}")

---
## 5. DGP True Parameters and Systematic Utilities

These are the **true DGP parameters** from §7 of the spec. Sourced from:
- β_time per mode: Ilahi et al. (2021) Table 11 VTTS → β_time = β_cost × VTTS / 60,000
- β_cost: Ilahi et al. Table 10: −1.42 per Thousand IDR
- ASCs: KRL=0 reference; others Bodetabek-adjusted per §7
- ρ (NL): Bastarianto et al. (2019) Table 3 + derived

In [44]:
# True DGP — from §7 MNL DGP
TRUE_DGP = {
    "asc": {
        "krl": 0.00, "car": 0.90, "moto": 1.20, "2wrh": 1.10,
        "4wrh": 0.50, "mrt": 0.30, "royal": 0.05, "lrt": -0.10, "tj": -0.30,
    },
    "beta_time": {
        "car": -0.60, "moto": -2.34, "4wrh": -3.49, "2wrh": -5.10,
        "krl": -2.72, "tj": -1.07, "royal": -1.30, "lrt": -2.37, "mrt": -2.98,
    },
    "beta_cost": -1.42,
}

TRUE_DGP_NL = {
    "rho_own_vehicle": 0.55,   # Bastarianto λ_hwh=0.55
    "rho_ridehailing": 0.70,   # Midpoint derived
    "rho_transit":     0.75,   # Bastarianto CNL cross-nest
}

print(f"DGP loaded: {len(TRUE_DGP['asc'])} ASCs + {len(TRUE_DGP['beta_time'])} β_time + β_cost")
print(f"Total MNL params: {len(TRUE_DGP['asc']) + len(TRUE_DGP['beta_time']) + 1}")
print(f"NL nest ρ: {TRUE_DGP_NL}")

DGP loaded: 9 ASCs + 9 β_time + β_cost
Total MNL params: 19
NL nest ρ: {'rho_own_vehicle': 0.55, 'rho_ridehailing': 0.7, 'rho_transit': 0.75}


In [45]:
# Compute V_m for each person-mode
for mode in MODE_LABELS:
    t = df_persons[f"time_{mode}"].values
    c = df_persons[f"cost_{mode}"].values
    
    v = np.full(N_PERSONS, -np.inf)
    mask = ~np.isnan(t) & ~np.isnan(c)
    v[mask] = (TRUE_DGP["asc"][mode]
               + TRUE_DGP["beta_time"][mode] * t[mask]
               + TRUE_DGP["beta_cost"] * c[mask])
    df_persons[f"V_{mode}"] = v

# Show sample
sample = df_persons.groupby("zone_id").first().reset_index()
v_cols = [f"V_{m}" for m in MODE_LABELS]
print("Sample V_m per zone (one person):\n")
print(sample[["zone_id", "income_segment"] + v_cols].to_string(
    float_format=lambda x: f"{x:7.1f}" if abs(x) < 1000 else "  -inf "))

Sample V_m per zone (one person):

  zone_id income_segment   V_car  V_moto  V_4wrh  V_2wrh   V_krl    V_tj  V_royal   V_lrt   V_mrt
0     J1a            low  -312.5  -222.6  -612.1  -570.2  -218.6   -inf     -inf    -inf    -inf 
1     J1b           high  -230.4  -154.8  -434.3  -405.5   -inf    -inf     -inf    -inf    -inf 
2      J2            low  -157.1   -98.3  -286.3  -267.9  -102.9   -74.8    -93.6  -102.0   -inf 
3     J3a            mid  -182.7  -118.7  -339.7  -317.5  -117.6   -inf    -154.7   -inf    -inf 
4     J3b            mid  -199.2  -131.8  -374.3  -349.3   -inf    -96.2   -169.6   -inf    -inf 
5      J4            mid  -170.2  -108.9  -314.2  -293.7  -103.5   -80.2   -101.6   -inf    -inf 
6      J5            mid   -86.6   -43.1  -142.2  -133.7   -45.4   -37.4    -inf    -inf    -52.4


In [46]:
# Deterministic choice check (no Gumbel noise — argmax V)
v_matrix = df_persons[[f"V_{m}" for m in MODE_LABELS]].values
best_mode_idx = np.argmax(v_matrix, axis=1)
best_mode = [MODE_LABELS[i] for i in best_mode_idx]

print("Deterministic mode choice by zone (no Gumbel noise):\n")
choice_summary = (
    pd.DataFrame({"zone_id": df_persons["zone_id"], "best_mode": best_mode})
    .groupby("zone_id")["best_mode"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
# Show only modes with > 0 share
print(choice_summary.to_string(float_format=lambda x: f"{x:.2f}"))
print("\nNote: deterministic (no noise). 02 notebook adds Gumbel → MLE recovery.")

Deterministic mode choice by zone (no Gumbel noise):

best_mode  krl  moto   tj
zone_id                  
J1a       1.00  0.00 0.00
J1b       0.00  1.00 0.00
J2        0.00  0.00 1.00
J3a       1.00  0.00 0.00
J3b       0.00  0.00 1.00
J4        0.00  0.00 1.00
J5        0.00  0.00 1.00

Note: deterministic (no noise). 02 notebook adds Gumbel → MLE recovery.


In [47]:
# Export
df_persons.to_csv(DATA_DIR / "persons_jkt.csv", index=False)
print(f"✅ Exported: {DATA_DIR / 'persons_jkt.csv'}")
print(f"   {len(df_persons)} rows × {len(df_persons.columns)} cols")
print(f"   {(DATA_DIR / 'persons_jkt.csv').stat().st_size / 1024:.1f} KB")

✅ Exported: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/persons_jkt.csv
   5000 rows × 33 cols
   939.5 KB


---
## 6. Summary and Verification

### Data lineage

| Value | Source | Type |
|---|---|---|
| Zone populations | `kelurahan_scores.geojson` — summed admin pop, scaled to commuting | Real data, proportionally scaled |
| Income per zone | `avg_household_expenditure` / 0.70 (SUSENAS ratio) | Real data |
| Income segments | Ilahi et al. (2021) Table 6: low 33.3% / mid 50.3% / high 16.4% | Literature ✓ |
| Vehicle ownership | Ilahi et al. (2021) Table 3: car 25.60% / MC 67.90% overall, by income segment | Literature ✓ |
| Poverty rate | `poverty_rate` — mean per zone | Real data |
| Distance to CBD | `distance_to_sudirman_km` — mean per zone | Real data |
| Car gen. cost | `gc_car_idr` — pipeline BPR calculation | Real data (computed) |
| Moto gen. cost | `gc_motorcycle_idr` — pipeline BPR calculation | Real data (computed) |
| Transit times | Schedule-based estimates (r5py NULL — documented gap) | Estimated, anchored to published timetables |
| Transit fares | Published fare tables (KCI, TJ, MRT Jakarta) | Real data |
| Ridehailing cost | Published tariff schedule × zone distance | Real tariff, estimated distance |
| Mode availability | `transit_stops_summary.csv` — spatial join to zones | Real data |
| GTFS headways | `frequencies.txt` or `transit_stops_summary.csv` | Real data |
| β_time, β_cost, ASCs | Ilahi et al. (2021) + Bastarianto et al. (2019) | Literature (DGP input) |

In [48]:
# Final verification
print("=" * 55)
print("FINAL VERIFICATION")
print("=" * 55)

files = ["jabodetabek_zones.csv", "od_skim_jkt.csv", "persons_jkt.csv"]
all_ok = True
for f in files:
    ok = (DATA_DIR / f).exists()
    if not ok:
        all_ok = False
    print(f"  {'✅' if ok else '❌'} {f}")

zones = pd.read_csv(DATA_DIR / "jabodetabek_zones.csv")
skim = pd.read_csv(DATA_DIR / "od_skim_jkt.csv")
persons = pd.read_csv(DATA_DIR / "persons_jkt.csv")

assert zones["population"].sum() > 6_000_000, "Population too low"
assert skim["cost_idr_thousand"].min() > 0, "Negative cost"
assert len(persons) == N_PERSONS, "Person count wrong"
assert persons["income_rp_k"].notna().all(), "NaN income"

for s, target in [("low", 0.333), ("mid", 0.503), ("high", 0.164)]:
    actual = (persons["income_segment"] == s).mean()
    assert abs(actual - target) < 0.03, f"{s} share off"

# Unavailable modes → -inf
for mode in MODE_LABELS:
    for z in ZONE_ORDER:
        if not df_avail.loc[z, mode]:
            z_mask = persons["zone_id"] == z
            vals = persons.loc[z_mask, f"V_{mode}"]
            assert (np.isneginf(vals.values)).all() or vals.isna().all(), \
                f"{z}×{mode} should be -inf"

print(f"\n{'='*55}")
print("ALL CHECKS PASSED — ready for 02_mnl_estimation.ipynb")
print(f"{'='*55}")
print(f"\nOutput files:")
print(f"  {DATA_DIR / 'jabodetabek_zones.csv'}: {len(zones)} zones")
print(f"  {DATA_DIR / 'od_skim_jkt.csv'}: {len(skim)} OD×mode pairs")
print(f"  {DATA_DIR / 'persons_jkt.csv'}: {len(persons)} persons")

FINAL VERIFICATION
  ✅ jabodetabek_zones.csv
  ✅ od_skim_jkt.csv
  ✅ persons_jkt.csv

ALL CHECKS PASSED — ready for 02_mnl_estimation.ipynb

Output files:
  /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/jabodetabek_zones.csv: 7 zones
  /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/od_skim_jkt.csv: 43 OD×mode pairs
  /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/persons_jkt.csv: 5000 persons


---
## Next: `02_mnl_estimation.ipynb`

The MNL estimation notebook will:
1. Read `data/persons_jkt.csv`
2. Generate synthetic choices: add Gumbel(0,1) noise to V_m, choose argmax
3. Estimate 18 MNL parameters via scipy MLE (reuse cells 13–23 from `logit_eda_mle.ipynb`)
4. Verify all parameters recover within 2 SE
5. Demonstrate IIA violation

Then → `03_nl_estimation.ipynb` — Nested Logit with 3 nests.